# L10: A safe baseline and predictive pipeline

Use training-side CV for tuning, then one final test evaluation. Synthetic target is not a deployed medical or business score.

**Environment:** See ../requirements.txt and the labs index. Examples use bundled inputs; no internet is required after dependency installation. Synthetic data is for teaching only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / "datasets").exists() else Path.cwd().parent
assert (ROOT / "datasets").exists(), "Run from the repository root or labs directory; in Colab upload/clone the repository first."
DATA = ROOT / "datasets"


## Independent synthetic regression and untouched test partition

In [ ]:
from sklearn.model_selection import train_test_split,GridSearchCV,KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error
rng=np.random.default_rng(42)
x=rng.uniform(-3,3,240).reshape(-1,1)
y=.7*x[:,0]**2-.5*x[:,0]+1+rng.normal(0,.65,240)
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=.25,random_state=399)
pipe=Pipeline([("imputer",SimpleImputer()),("poly",PolynomialFeatures(include_bias=False)),("scale",StandardScaler()),("model",Ridge())])
search=GridSearchCV(pipe,{"poly__degree":[1,2,3],"model__alpha":[.01,1,10]},cv=KFold(5,shuffle=True,random_state=42),scoring="neg_mean_squared_error")
search.fit(X_train,y_train)
print("Chosen by training-only CV:",search.best_params_)
# GridSearchCV refits the chosen pipeline on all training rows.
baseline=DummyRegressor().fit(X_train,y_train)
for name,model in [("Baseline",baseline),("Chosen pipeline",search.best_estimator_)]:
    prediction=model.predict(X_test)
    print(name,"MSE",mean_squared_error(y_test,prediction),"MAE",mean_absolute_error(y_test,prediction))

## Student task
Explain why each preprocessing component is inside the pipeline. Define MSE units. Report the final score without changing the recipe in response to the test result.

## Interpretation and limitations
Record the dataset grain, units, assumptions, and a limitation in your submission. Explain one result rather than only submitting screenshots.